In [ ]:
# stock_forecast_app.py

import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import plotly.graph_objs as go
from datetime import datetime

# ------------------ SETUP ------------------
st.set_page_config(page_title="Stock Price Prediction", layout="wide")
st.title("📈 Stock Price Prediction (2010–2025)")
st.sidebar.header("Model Settings")

ticker = "AAPL"
start_date = "2010-01-01"
end_date = "2025-01-01"
forecast_years = st.sidebar.slider("Years to Forecast", 1, 5, 2)
forecast_days = forecast_years * 252  # Roughly trading days in a year
model_type = st.sidebar.selectbox("Select Model", ["ARIMA", "SARIMA", "Prophet", "LSTM"])

# ------------------ DATA ------------------
@st.cache_data
def load_data(ticker):
    df = yf.download(ticker, start=start_date, end=end_date)
    df.reset_index(inplace=True)
    return df

df = load_data(ticker)
df = df[['Date', 'Close']]
df.columns = ['ds', 'y']  # Prophet-compatible

st.subheader(f"Raw Data for {ticker}")
st.line_chart(df.set_index('ds')['y'])

# ------------------ MODELS ------------------

def arima_model(data, forecast_days):
    from pandas.plotting import register_matplotlib_converters
    register_matplotlib_converters()
    ts = data.set_index('ds')['y']
    model = ARIMA(ts, order=(5, 1, 0))
    model_fit = model.fit()
    forecast = model_fit.forecast(steps=forecast_days)
    forecast_dates = pd.date_range(data['ds'].iloc[-1], periods=forecast_days+1, freq='B')[1:]
    return pd.DataFrame({'ds': forecast_dates, 'y': forecast})

def sarima_model(data, forecast_days):
    ts = data.set_index('ds')['y']
    model = SARIMAX(ts, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12))
    model_fit = model.fit(disp=False)
    forecast = model_fit.forecast(steps=forecast_days)
    forecast_dates = pd.date_range(data['ds'].iloc[-1], periods=forecast_days+1, freq='B')[1:]
    return pd.DataFrame({'ds': forecast_dates, 'y': forecast})

def prophet_model(data, forecast_days):
    m = Prophet()
    m.fit(data)
    future = m.make_future_dataframe(periods=forecast_days)
    forecast = m.predict(future)
    return forecast[['ds', 'yhat']].rename(columns={'yhat': 'y'}).tail(forecast_days)

def lstm_model(data, forecast_days):
    df_lstm = data.copy()
    df_lstm.set_index('ds', inplace=True)
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df_lstm)

    sequence_length = 60
    X, y = [], []
    for i in range(sequence_length, len(scaled_data)):
        X.append(scaled_data[i - sequence_length:i, 0])
        y.append(scaled_data[i, 0])
    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=(X.shape[1], 1)))
    model.add(Dropout(0.2))
    model.add(LSTM(50))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X, y, epochs=5, batch_size=32, verbose=0)

    input_seq = scaled_data[-sequence_length:]
    forecast = []
    for _ in range(forecast_days):
        input_data = input_seq[-sequence_length:].reshape(1, sequence_length, 1)
        predicted = model.predict(input_data, verbose=0)
        forecast.append(predicted[0, 0])
        input_seq = np.append(input_seq, predicted, axis=0)

    forecast_scaled = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    forecast_dates = pd.date_range(data['ds'].iloc[-1], periods=forecast_days+1, freq='B')[1:]
    return pd.DataFrame({'ds': forecast_dates, 'y': forecast_scaled})

# ------------------ FORECASTING ------------------

st.subheader(f"{model_type} Forecast for {forecast_years} Years")

with st.spinner("Training and forecasting..."):
    if model_type == "ARIMA":
        forecast_df = arima_model(df, forecast_days)
    elif model_type == "SARIMA":
        forecast_df = sarima_model(df, forecast_days)
    elif model_type == "Prophet":
        forecast_df = prophet_model(df, forecast_days)
    elif model_type == "LSTM":
        forecast_df = lstm_model(df, forecast_days)

# ------------------ PLOTTING ------------------

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['ds'], y=df['y'], mode='lines', name='Historical'))
fig.add_trace(go.Scatter(x=forecast_df['ds'], y=forecast_df['y'], mode='lines', name='Forecast'))
fig.update_layout(title=f"{model_type} Forecast for {ticker}", xaxis_title="Date", yaxis_title="Price")
st.plotly_chart(fig, use_container_width=True)


C:\Users\Asus Tuf\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2025-07-21 12:05:04.146 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-21 12:05:04.146 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-21 12:05:06.571 
  command:

    streamlit run C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-07-21 12:05:06.571 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-21 12:05:06.571 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.

DeltaGenerator()

['Requirement already satisfied: plotly in c:\\users\\asus tuf\\.conda\\envs\\cleanenv\\lib\\site-packages (6.2.0)',
 'Requirement already satisfied: narwhals>=1.15.1 in c:\\users\\asus tuf\\.conda\\envs\\cleanenv\\lib\\site-packages (from plotly) (1.47.1)',
 'Requirement already satisfied: packaging in c:\\users\\asus tuf\\appdata\\roaming\\python\\python310\\site-packages (from plotly) (24.2)']